In [1]:
#1
# ====== CELL ĐẦU TIÊN — chạy ngay sau khi Restart Kernel, trước mọi cell khác ======
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"   # phải set TRƯỚC khi bất kỳ package HF nào được import

!pip uninstall -y hf_xet hf-xet -q
!pip install -q open_clip_torch peft sentence-transformers

print("Đã tắt Xet CDN, sẵn sàng chạy tiếp các cell load model.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 84.9 MB/s eta 0:00:00:00:01
Đã tắt Xet CDN, sẵn sàng chạy tiếp các cell load model.


In [2]:
#2
# ====== FIX: gỡ hf_xet để tránh lỗi 403 khi tải qua Xet CDN ======
!pip uninstall -y hf_xet hf-xet
!pip install -U torchao

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch
import open_clip

# ====== CONFIG ======
MODEL_NAME = "ViT-B-32-quickgelu"
PRETRAINED = "openai"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ====== LOAD CLIP MODEL ======
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model = model.to(DEVICE)

# ====== ĐÓNG BĂNG IMAGE ENCODER ======
for param in model.visual.parameters():
    param.requires_grad = False
model.visual.eval()

# ====== CHECK LẠI ======
total_params = sum(p.numel() for p in model.visual.parameters())
frozen_params = sum(p.numel() for p in model.visual.parameters() if not p.requires_grad)
trainable_params = total_params - frozen_params

print(f"Tổng tham số Image Encoder: {total_params:,}")
print(f"Đã đóng băng: {frozen_params:,}")
print(f"Còn train được: {trainable_params:,}")

assert trainable_params == 0, "Image Encoder chưa được đóng băng hoàn toàn!"
print("\n✅ Image Encoder đã đóng băng thành công.")

Found existing installation: hf-xet 1.5.1
Uninstalling hf-xet-1.5.1:
  Successfully uninstalled hf-xet-1.5.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.4 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


open_clip_pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Tổng tham số Image Encoder: 87,849,216
Đã đóng băng: 87,849,216
Còn train được: 0

✅ Image Encoder đã đóng băng thành công.


In [3]:
#3
# 1. Cài đặt thư viện
!pip install -q peft sentence-transformers

# 2. Tải model từ Hugging Face về Kaggle Working
!git clone https://huggingface.co/sentence-transformers/clip-ViT-B-32-multilingual-v1 /kaggle/working/clip-multilingual

# 3. Cấu hình môi trường và Import
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch
from sentence_transformers import SentenceTransformer
from peft import LoraConfig, get_peft_model

# 4. Thiết lập Device và Load model từ đường dẫn local
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Gọi model sau khi đã import thành công
mclip = SentenceTransformer("/kaggle/working/clip-multilingual")
mclip = mclip.to(DEVICE)

print("✅ Đã load model thành công từ Local!")

Cloning into '/kaggle/working/clip-multilingual'...
remote: Enumerating objects: 91, done.
remote: Total 91 (delta 0), reused 0 (delta 0), pack-reused 91 (from 1)
Receiving objects: 100% (91/91), 1.41 MiB | 19.58 MiB/s, done.
Resolving deltas: 100% (31/31), done.
Filtering content: 100% (16/16), 4.90 GiB | 99.66 MiB/s, done.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

✅ Đã load model thành công từ Local!


In [4]:
#4
!pip install -U torchao

In [5]:
#5
# ====== 2. Gắn LoRA vào Transformer (DistilBERT-multilingual, 6 layer) ======
transformer_module = mclip[0].auto_model   # lấy DistilBertModel bên trong
dense_module = mclip[2]                     # lớp Dense cuối (projection 768 -> 512)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_lin", "k_lin", "v_lin", "out_lin"],   # tên layer attention của DistilBERT
)

peft_transformer = get_peft_model(transformer_module, lora_config)
mclip[0].auto_model = peft_transformer   # gắn ngược lại vào SentenceTransformer

# ====== 3. Mở khóa riêng lớp Dense cuối (train full, không qua LoRA) ======
for param in dense_module.parameters():
    param.requires_grad = True

# ====== 4. Check số tham số train được ======
total_params = sum(p.numel() for p in mclip.parameters())
trainable_params = sum(p.numel() for p in mclip.parameters() if p.requires_grad)

print(f"\nTổng tham số toàn bộ Multilingual Text Encoder: {total_params:,}")
print(f"Có thể train (LoRA + Dense cuối): {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

peft_transformer.print_trainable_parameters()


Tổng tham số toàn bộ Multilingual Text Encoder: 135,717,120
Có thể train (LoRA + Dense cuối): 983,040 (0.72%)
trainable params: 589,824 || all params: 135,323,904 || trainable%: 0.4359


In [6]:
#6
import os
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ====== 0. Đọc CSV ======
CSV_PATH = "/kaggle/input/datasets/js042710/final-datasets-tolora/final_coco_19k_dataset.csv"   # <-- sửa lại đúng path của bạn
df = pd.read_csv(CSV_PATH)
df["full_path"] = df["image_dir"].str.rstrip("/") + "/" + df["file_name"]

# ====== 1. Lấy danh sách ảnh unique (mỗi ảnh chỉ encode 1 lần dù có nhiều caption) ======
unique_images = df.drop_duplicates(subset="image_id")[["image_id", "full_path"]].reset_index(drop=True)
print(f"Số ảnh unique cần encode: {len(unique_images)}")

class ImageOnlyDataset(Dataset):
    def __init__(self, image_ids, image_paths, preprocess):
        self.image_ids = image_ids
        self.image_paths = image_paths
        self.preprocess = preprocess

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        return self.preprocess(img), self.image_ids[idx]

img_dataset = ImageOnlyDataset(
    unique_images["image_id"].tolist(),
    unique_images["full_path"].tolist(),
    preprocess,   # đã có sẵn từ cell load CLIP ban đầu
)
img_loader = DataLoader(img_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# ====== 2. Forward qua image encoder đóng băng (no_grad) ======
image_embeddings = {}   # image_id -> tensor 512-dim

model.visual.eval()
with torch.no_grad():
    for imgs, ids in tqdm(img_loader, desc="Encoding images"):
        imgs = imgs.to(DEVICE)
        feats = model.visual(imgs).float().cpu()
        for i, img_id in enumerate(ids):
            image_embeddings[int(img_id)] = feats[i]

print(f"Đã encode xong {len(image_embeddings)} ảnh.")
torch.save(image_embeddings, "/kaggle/working/image_embeddings.pt")

Số ảnh unique cần encode: 19000


Encoding images:   0%|          | 0/75 [00:00<?, ?it/s]

Đã encode xong 19000 ảnh.


In [7]:
#7
import numpy as np

# ====== 0. Loại bỏ các dòng có image_id không có embedding ======
valid_ids = set(image_embeddings.keys())
df = df[df["image_id"].isin(valid_ids)].reset_index(drop=True)
print(f"Số sample hợp lệ sau lọc: {len(df)}")

# ====== 1. Split theo image_id (tránh leakage: 1 ảnh không được nằm cả 2 tập) ======
rng = np.random.RandomState(42)
unique_ids = df["image_id"].unique()
rng.shuffle(unique_ids)

n_val = max(1, int(len(unique_ids) * 0.05))   # 5% ảnh làm validation
val_ids = set(unique_ids[:n_val])
train_ids = set(unique_ids[n_val:])

train_df = df[df["image_id"].isin(train_ids)].reset_index(drop=True)
val_df = df[df["image_id"].isin(val_ids)].reset_index(drop=True)

print(f"Train: {len(train_df)} caption | {len(train_ids)} ảnh")
print(f"Val:   {len(val_df)} caption | {len(val_ids)} ảnh")

# ====== 2. Tokenizer lấy sẵn từ mclip ======
tokenizer = mclip[0].tokenizer

class ImageTextDataset(Dataset):
    def __init__(self, df, image_embeddings):
        self.image_ids = df["image_id"].tolist()
        self.captions = df["caption"].astype(str).tolist()
        self.image_embeddings = image_embeddings

    def __len__(self):
        return len(self.captions)

    def __getitem__(self, idx):
        return self.image_embeddings[int(self.image_ids[idx])], self.captions[idx]

def collate_fn(batch):
    img_embeds, captions = zip(*batch)
    img_embeds = torch.stack(img_embeds)
    text_inputs = tokenizer(list(captions), padding=True, truncation=True, max_length=77, return_tensors="pt")
    return img_embeds, text_inputs

train_dataset = ImageTextDataset(train_df, image_embeddings)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2,
                           collate_fn=collate_fn, drop_last=True)
print(f"Số batch/epoch (train): {len(train_loader)}")

Số sample hợp lệ sau lọc: 135091
Train: 128340 caption | 18050 ảnh
Val:   6751 caption | 950 ảnh
Số batch/epoch (train): 1002


In [8]:
#8
@torch.no_grad()
def evaluate_recall(mclip, val_df, image_embeddings, device, k_list=(1, 5, 10)):
    mclip.eval()

    # Mỗi ảnh unique trong val chỉ encode 1 lần
    val_unique_ids = val_df["image_id"].unique().tolist()
    id_to_idx = {img_id: i for i, img_id in enumerate(val_unique_ids)}

    image_feats = torch.stack([image_embeddings[i] for i in val_unique_ids]).to(device)
    image_feats = F.normalize(image_feats, dim=-1)

    captions = val_df["caption"].astype(str).tolist()
    caption_img_idx = [id_to_idx[i] for i in val_df["image_id"].tolist()]

    text_feats = []
    batch_size = 128
    for i in range(0, len(captions), batch_size):
        batch_caps = captions[i:i+batch_size]
        text_inputs = tokenizer(batch_caps, padding=True, truncation=True, max_length=77, return_tensors="pt")
        text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
        features = dict(text_inputs)
        for module in mclip:
            features = module(features)
        text_feats.append(features["sentence_embedding"].cpu())
    text_feats = torch.cat(text_feats).to(device)
    text_feats = F.normalize(text_feats, dim=-1)

    caption_img_idx = torch.tensor(caption_img_idx, device=device)

    # ---- Text -> Image Recall@K ----
    sim_t2i = text_feats @ image_feats.t()          # (n_captions, n_images)
    ranks_t2i = sim_t2i.argsort(dim=-1, descending=True)
    results = {}
    for k in k_list:
        topk = ranks_t2i[:, :k]
        correct = (topk == caption_img_idx.unsqueeze(1)).any(dim=1).float().mean().item()
        results[f"text2image_R@{k}"] = correct * 100

    # ---- Image -> Text Recall@K (mỗi ảnh coi đúng nếu match bất kỳ caption nào của chính nó) ----
    sim_i2t = image_feats @ text_feats.t()          # (n_images, n_captions)
    ranks_i2t = sim_i2t.argsort(dim=-1, descending=True)
    caption_img_idx_np = caption_img_idx.cpu().numpy()
    for k in k_list:
        correct = 0
        topk = ranks_i2t[:, :k].cpu().numpy()
        for img_idx in range(len(val_unique_ids)):
            gt_caption_idxs = np.where(caption_img_idx_np == img_idx)[0]
            if len(set(topk[img_idx]) & set(gt_caption_idxs)) > 0:
                correct += 1
        results[f"image2text_R@{k}"] = correct / len(val_unique_ids) * 100

    mclip.train()
    return results

In [9]:
#9
import torch.nn.functional as F
from torch.optim import AdamW

# ====== 1. logit_scale học được, khởi tạo giống CLIP gốc: ln(1/0.07) ======
logit_scale = torch.nn.Parameter(torch.tensor(np.log(1/0.07), device=DEVICE))

# ====== 2. Optimizer: chỉ update LoRA + Dense cuối + logit_scale ======
trainable_params = [p for p in mclip.parameters() if p.requires_grad] + [logit_scale]
optimizer = AdamW(trainable_params, lr=1e-4, weight_decay=0.01)

EPOCHS = 5
scaler = torch.cuda.amp.GradScaler()

def clip_loss(image_embeds, text_embeds, logit_scale):
    image_embeds = F.normalize(image_embeds, dim=-1)
    text_embeds = F.normalize(text_embeds, dim=-1)
    scale = logit_scale.exp().clamp(max=100)
    logits_per_text = scale * text_embeds @ image_embeds.t()
    logits_per_image = logits_per_text.t()
    labels = torch.arange(image_embeds.size(0), device=image_embeds.device)
    loss_i = F.cross_entropy(logits_per_image, labels)
    loss_t = F.cross_entropy(logits_per_text, labels)
    return (loss_i + loss_t) / 2

# ====== 3. Training loop ======
mclip.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for img_embeds, text_inputs in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        img_embeds = img_embeds.to(DEVICE)
        text_inputs = {k: v.to(DEVICE) for k, v in text_inputs.items()}

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            features = dict(text_inputs)
            for module in mclip:
                features = module(features)
            text_embeds = features["sentence_embedding"]
            loss = clip_loss(img_embeds, text_embeds, logit_scale)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    metrics = evaluate_recall(mclip, val_df, image_embeddings, DEVICE)
    metrics_str = " | ".join(f"{k}={v:.2f}" for k, v in metrics.items())
    print(f"Epoch {epoch+1}: loss = {avg_loss:.4f} | logit_scale = {logit_scale.exp().item():.2f}")
    print(f"  → {metrics_str}")

/tmp/ipykernel_58/2603483979.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 1/5:   0%|          | 0/1002 [00:00<?, ?it/s]

/tmp/ipykernel_58/2603483979.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1: loss = 1.2549 | logit_scale = 15.62
  → text2image_R@1=44.87 | text2image_R@5=77.22 | text2image_R@10=88.80 | image2text_R@1=68.32 | image2text_R@5=89.05 | image2text_R@10=94.00


Epoch 2/5:   0%|          | 0/1002 [00:00<?, ?it/s]

...

/tmp/ipykernel_58/2603483979.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 4: loss = 0.7422 | logit_scale = 20.13
  → text2image_R@1=50.24 | text2image_R@5=81.59 | text2image_R@10=91.76 | image2text_R@1=68.84 | image2text_R@5=90.11 | image2text_R@10=94.74


Epoch 5/5:   0%|          | 0/1002 [00:00<?, ?it/s]

/tmp/ipykernel_58/2603483979.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 5: loss = 0.6583 | logit_scale = 21.83
  → text2image_R@1=51.24 | text2image_R@5=82.37 | text2image_R@10=92.09 | image2text_R@1=69.47 | image2text_R@5=91.05 | image2text_R@10=95.37


In [10]:
import os

SAVE_DIR = "/kaggle/working/finetuned_mclip"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Lưu trọng số LoRA của bộ Text Encoder
mclip[0].auto_model.save_pretrained(f"{SAVE_DIR}/lora_weights")

# 2. Lưu trọng số của lớp Dense cuối (do mình mở khóa train full lớp này)
torch.save(mclip[2].state_dict(), f"{SAVE_DIR}/dense_weights.pth")

print(f"✅ Đã lưu trọng số thành công tại: {SAVE_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Đã lưu trọng số thành công tại: /kaggle/working/finetuned_mclip
